# A Significant Result Is Not a Replicable Effect

This executable companion demonstrates power, conditional Type S and Type M errors, the winner's curse, independent replication, FDR control, prior-dependent shrinkage, and conditional confidence-interval coverage. All data are synthetic.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
from analysis import (Design, critical_value, two_sided_power, type_s_error,
                      type_m_error, significant_interval_coverage,
                      benjamini_hochberg, one_multiple_testing_family,
                      posterior_mean_normal, run_study)
cfg = Design()
delta = cfg.true_effect / cfg.study_se
print('Design:', cfg)
print('Standardized effect:', delta)

In [ ]:
print('Two-sided power:', round(two_sided_power(delta), 4))
print('Wrong-sign rate among selected:', round(type_s_error(delta), 4))
print('Absolute exaggeration among selected:', round(type_m_error(delta), 4))
assert 0.16 < two_sided_power(delta) < 0.18
assert type_m_error(delta) > 2

In [ ]:
print('Usual unconditional CI coverage:', 1-cfg.alpha)
print('Coverage given significance:', round(significant_interval_coverage(delta), 4))
assert 0.80 < significant_interval_coverage(delta) < 0.90

In [ ]:
rng = np.random.default_rng(cfg.seed)
first = cfg.true_effect + cfg.study_se * rng.standard_normal(20000)
selected = np.abs(first / cfg.study_se) > critical_value()
replication = cfg.true_effect + cfg.study_se * rng.standard_normal(20000)
print('Mean reported initial estimate:', round(float(first[selected].mean()), 4))
print('Mean independent replication:', round(float(replication[selected].mean()), 4))
assert abs(float(replication[selected].mean()) - cfg.true_effect) < 0.03

In [ ]:
example = one_multiple_testing_family(np.random.default_rng(200), m=1000, pi0=.95)
for method, result in example.items():
    print(method, 'discoveries:', int(result['discoveries']),
          'false discoveries:', int(result['false_discoveries']))
assert example['benjamini_hochberg']['discoveries'] <= example['uncorrected']['discoveries']

In [ ]:
reported = np.array([-.9, -.6, .6, .9], dtype=float)
shrunk = posterior_mean_normal(reported, cfg.study_se, cfg.shrinkage_prior_sd)
print('Published:', reported, 'Posterior means with stated Normal prior:', shrunk)
assert np.all(np.abs(shrunk) < np.abs(reported))

In [ ]:
with TemporaryDirectory() as location:
    outputs = run_study(quick=True, output_dir=Path(location))
    assert len(list((Path(location) / 'figures').glob('*_quick.png'))) == 8
    assert len(list((Path(location) / 'results').glob('*_quick.csv'))) == 10
    assert outputs['type_m_at_delta_1'] > 2
    assert outputs['bh_fdr_pi0_0_95'] < outputs['uncorrected_fdr_pi0_0_95']
    print('Quick run: 8 figures, 10 CSV tables, seeded and reproducible')

## Interpretation

Significance filtering conditions on a stochastic selection event; unconditional unbiasedness and nominal interval coverage need not survive it. The Bayesian shrinkage comparison assumes the stated prior is correct. BH FDR control here assumes independent hypothesis tests. Replication is independently sampled and unfiltered. See the full manuscript and README for limits and references.